# Assignment 6 - Klasifikasi Defect NEU

Alur representasi gambar mengadaptasi konsep pada **Modul 7** dan **Modul 8** dalam PDF praktikum: piksel gambar diratakan menjadi vektor untuk klasifikasi KNN, lalu PCA di-fit pada data training dan dipakai untuk transformasi fitur. Modul 7 membahas KNN supervised; Modul 8 membahas PCA/eigenfaces sebagai preprocessing. PCA di sini dipakai sebagai reduksi dimensi untuk KNN dan SVM, sesuai tujuan klasifikasi Assignment 6.

Pipeline tambahan memisahkan crop berdasarkan gambar agar crop dari gambar yang sama tidak bocor ke split atau fold yang berbeda.

## 1. Konfigurasi dan dependensi

Dependensi: `opencv-python`, `numpy`, `pandas`, `matplotlib`, `PyYAML`, dan `scikit-learn`. Cell konfigurasi akan memasang dependensi yang belum tersedia.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

packages = {
    "cv2": "opencv-python",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "yaml": "PyYAML",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import pandas as pd
from IPython.display import display

candidates = [Path.cwd() / "assignment_6", Path.cwd()]
candidates.extend(Path.cwd().parents)
ASSIGNMENT_DIR = next(
    (path.resolve() for path in candidates if (path / "assignment6_pipeline.py").is_file()),
    None,
)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError("Tidak menemukan assignment6_pipeline.py. Pastikan folder assignment_6 tersedia.")

sys.path.insert(0, str(ASSIGNMENT_DIR))
from assignment6_pipeline import run

DATASET_DIR = ASSIGNMENT_DIR.parent / "project_akhir_yolov8" / "Defect in NEU.v2i.yolov8"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs" / "assignment6_run"
print("Folder assignment:", ASSIGNMENT_DIR)
print("Dataset:", DATASET_DIR)
print("Output:", OUTPUT_DIR)

Folder assignment: C:\Users\user\Downloads\NEUDefect\assignment_6
Dataset: C:\Users\user\Downloads\NEUDefect\project_akhir_yolov8\Defect in NEU.v2i.yolov8
Output: C:\Users\user\Downloads\NEUDefect\assignment_6\outputs\assignment6_run


## 2. Audit data, split, training, dan evaluasi

Alurnya mengikuti materi modul: Modul 7 meratakan piksel gambar menjadi vektor untuk KNN; Modul 8 menggunakan PCA pada data train sebelum transformasi. Di sini crop grayscale 32 x 32 diratakan menjadi vektor piksel, lalu direduksi dengan PCA 150 komponen (`whiten=True`). Dataset dibagi 80/20 per gambar secara stratified, dan 5-fold CV dikelompokkan per gambar untuk mencegah kebocoran antar-crop. Oversampling KNN dan pemilihan parameter hanya memakai data training/development.

In [2]:
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_DIR}. Ubah DATASET_DIR sesuai lokasi data.")

results = run(DATASET_DIR, OUTPUT_DIR)
print("\nDistribusi crop:")
display(results["class_distribution"])
print("\nHasil 5-fold cross-validation:")
display(results["cv_summary"])
print("\nEvaluasi akhir pada test set:")
display(results["test_metrics"])


Dataset: 1799 images ditemukan, 1799 usable, 4186 bounding boxes, 0 duplicate dilewati.
Split gambar: development 1439 (80.0%), test 360 (20.0%).
Fitur crop: development=3323, test=863.

Distribusi crop sebelum balancing train:
          class  development_crops  test_crops
        crazing                543         146
      inclusion                806         205
        patches                701         177
 pitted_surface                341          91
rolled-in_scale                498         130
      scratches                434         114



Ringkasan 5-fold CV:
model                 configuration  accuracy_mean  accuracy_std  precision_macro_mean  precision_macro_std  recall_macro_mean  recall_macro_std  f1_macro_mean  f1_macro_std
  KNN                           k=1       0.403849      0.005218              0.520319             0.083456           0.379983          0.005929       0.326520      0.003607
  KNN                           k=3       0.390907      0.006441              0.487386             0.067018           0.369177          0.006456       0.304513      0.007596
  KNN                           k=5       0.377968      0.005175              0.436518             0.042239           0.355631          0.007153       0.285458      0.006231
  KNN                           k=7       0.362318      0.009570              0.442887             0.078000           0.335797          0.011116       0.268324      0.007467
  KNN                           k=9       0.350582      0.011232              0.440763             0.085358 


Hasil evaluasi test (20% holdout):
model                 configuration  accuracy  precision_macro  recall_macro  f1_macro
  KNN                           k=1  0.402086         0.609391      0.380941  0.329420
  SVM kernel=rbf,C=10.0,gamma=scale  0.778679         0.786308      0.761457  0.768486

Distribusi crop:


,class,development_crops,test_crops
0,crazing,543,146
1,inclusion,806,205
2,patches,701,177
3,pitted_surface,341,91
4,rolled-in_scale,498,130
5,scratches,434,114



Hasil 5-fold cross-validation:


,model,configuration,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,KNN,k=1,0.403849,0.005218,0.520319,0.083456,0.379983,0.005929,0.326520,0.003607
1,KNN,k=3,0.390907,0.006441,0.487386,0.067018,0.369177,0.006456,0.304513,0.007596
2,KNN,k=5,0.377968,0.005175,0.436518,0.042239,0.355631,0.007153,0.285458,0.006231
3,KNN,k=7,0.362318,0.009570,0.442887,0.078000,0.335797,0.011116,0.268324,0.007467
4,KNN,k=9,0.350582,0.011232,0.440763,0.085358,0.321519,0.013479,0.253831,0.013335
7,SVM,"kernel=rbf,C=10.0,gamma=scale",0.769168,0.021328,0.770106,0.023147,0.744620,0.024180,0.752646,0.023678
6,SVM,"kernel=rbf,C=1.0,gamma=scale",0.754137,0.013414,0.765057,0.012989,0.739222,0.014649,0.743126,0.013169
5,SVM,"kernel=linear,C=1.0",0.641278,0.014293,0.639936,0.013689,0.621407,0.013909,0.627059,0.013522



Evaluasi akhir pada test set:


,model,configuration,accuracy,precision_macro,recall_macro,f1_macro
0,KNN,k=1,0.402086,0.609391,0.380941,0.329420
1,SVM,"kernel=rbf,C=10.0,gamma=scale",0.778679,0.786308,0.761457,0.768486


## 3. Buat laporan PDF

Cell ini menyusun laporan dari output yang baru dibuat, termasuk tabel cross-validation, metrik test, dan confusion matrix.

In [3]:
from generate_report import generate_report
pdf_path = generate_report(OUTPUT_DIR, ASSIGNMENT_DIR / "Laporan_Assignment6.pdf")
print("Laporan PDF:", pdf_path)

Laporan PDF: C:\Users\user\Downloads\NEUDefect\assignment_6\Laporan_Assignment6.pdf


## 4. Berkas hasil

Semua output disimpan di `assignment_6/outputs/assignment6_run/`:

- `dataset_audit.csv` — status gambar dan label.
- `split_manifest.csv` — penetapan gambar ke development/test.
- `class_distribution.csv` — distribusi crop per kelas.
- `cv_fold_results.csv` dan `cv_summary.csv` — hasil 5-fold CV KNN/SVM.
- `test_metrics.csv` dan `classification_report_*.csv` — metrik test dan laporan per kelas.
- `confusion_matrix_*.csv` dan `confusion_matrix_*.png` — confusion matrix.
- `run_summary.txt` — ringkasan konfigurasi dan hasil.

Gunakan tabel dan grafik hasil yang benar-benar dihasilkan saat notebook dijalankan untuk menyusun laporan PDF. Jangan menyalin angka sebelum eksekusi berhasil.